# 03 — H&E/IHC slide alignment

`rp.align` registers moving slides (IHC) onto reference slides (H&E) and
exports each aligned slide in the reference's coordinates.

- `backend="orb"` — fast global affine alignment (extra: `orb`).
- `backend="valis"` — rigid and non-rigid, better for local deformation (extra: `valis`).

Pairs live in pair folders, with the sample ID shared by both filenames:

```text
data/pairs/
└── CD8/
    ├── he/   Sample_0001_he.svs   Sample_0002_he.svs
    └── cd8/  Sample_0001_cd8.svs  Sample_0002_cd8.svs
```

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
PAIRS_ROOT = DATA_ROOT / "pairs"
OUTPUT_DIR = RESULTS_ROOT / "aligned"

settings = dict(
    pair_folders=["CD8"],
    reference_name="he",
    moving_name="cd8",
    backend="orb",                         # or "valis"
    target_magnification=20.0,
    reference_source_magnification=None,   # set only when metadata is absent
    moving_source_magnification=None,
    qc_enabled=True,
)

RUN_DRY_RUN = False
RUN_ALIGNMENT = False

## Check pairing first

A dry run discovers and pairs slides without registering anything; read the
log for `[DRY RUN]` lines. The filename pattern it uses is derived from the
role names.

In [ ]:
print("Filename pattern:", rp.AlignConfig(**settings).filename_pattern)

if RUN_DRY_RUN:
    rp.align(PAIRS_ROOT, OUTPUT_DIR, **settings, dry_run=True)
else:
    print("Set RUN_DRY_RUN=True after editing the Parameters cell.")

## Align

Two slides can also be aligned directly: `rp.align([reference, moving], output_dir)`.
QC figures show the aligned slide centers; still review edges, vessels and
glands across the whole slide.

In [ ]:
if RUN_ALIGNMENT:
    aligned = rp.align(PAIRS_ROOT, OUTPUT_DIR, **settings)
    for item in aligned.by_role("aligned"):
        print(item.sample_id, item.path)
    for figure in aligned.by_role("figure"):
        print("QC:", figure.path.name)
else:
    print("Set RUN_ALIGNMENT=True once the dry run pairs correctly.")

## Outputs

```text
results/aligned/
├── rocqipath.json                              # what was aligned, and how
└── alignment/Sample_0001_cd8/
    ├── Sample_0001_cd8_aligned_moving.ome.tiff
    ├── Sample_0001_cd8_aligned_moving_manifest.json   # its magnification
    └── Sample_0001_cd8_center_qc.png
```

Later workflows take `results/aligned` (or the `aligned` result) directly —
see notebook **04**.

**Troubleshooting**: no pairs → check folder and role names; "filename did not
match" → filenames must end in the role token (`_he`, `_cd8`); physical field
ratio error → check magnifications; poor ORB match → try VALIS.